In [ ]:
# 셀 1: 라이브러리 설치
!pip install pandas openpyxl pymysql sqlalchemy

In [ ]:
# 셀 2: 임포트 & DB 연결 설정
import pandas as pd
import pymysql
from sqlalchemy import create_engine, text

# ✅ 본인 DB 정보로 수정
DB_HOST = 'mp.smhrd.or.kr'
DB_PORT = 3306
DB_USER = 'lang9_core_overfit'
DB_PASS = 'overfit9'
DB_NAME = 'lang9_core_overfit'

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4'
)

# 연결 테스트
with engine.connect() as conn:
    print('✅ DB 연결 성공!')

In [ ]:
# 셀 3: 데이터 로드 & 전처리
df = pd.read_excel('final.xlsx')

# 전성분 앞뒤 따옴표 제거
df['전성분'] = df['전성분'].astype(str).str.strip('"').str.strip()
df['전성분'] = df['전성분'].replace('nan', None)

# ad_flag 없는 값 N으로 채우기
df['ad_flag'] = df['ad_flag'].fillna('N')

print(f'총 리뷰: {len(df)}행')
print(f'고유 제품: {df["제품명"].nunique()}개')
df.head(2)

In [ ]:
# 셀 4 전체 교체
products_df = df[['제품명', '브랜드', '카테고리', '이미지URL']].drop_duplicates(subset='제품명').reset_index(drop=True)

products_insert = products_df.rename(columns={
    '제품명': 'prod_name',
    '브랜드': 'brand_name',
    '카테고리': 'category',
    '이미지URL': 'img_url'
})

products_insert['target_skin'] = None
products_insert = products_insert.dropna(subset=['brand_name']).reset_index(drop=True)
products_insert['category'] = products_insert['category'].fillna('기타')
products_insert['img_url'] = products_insert['img_url'].fillna('')

# commit 명시
with engine.begin() as conn:
    products_insert.to_sql(
        't_product',
        con=conn,
        if_exists='append',
        index=False,
        chunksize=500
    )

print(f'✅ t_product 적재 완료: {len(products_insert)}개')

# 바로 확인
with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM t_product')).scalar()
    print(f'DB t_product: {count}행')

In [ ]:
# 셀 5: prod_idx 매핑 (방금 적재된 제품 idx 가져오기)

with engine.connect() as conn:
    prod_map = pd.read_sql('SELECT prod_idx, prod_name FROM t_product', conn)

print(f't_product에서 {len(prod_map)}개 제품 로드')
prod_map.head()

In [ ]:
# 셀 6: t_review 적재

reviews_df = df[['제품명', '리뷰', '별점', 'ad_flag', '리뷰URL']].copy()

# prod_idx JOIN
reviews_df = reviews_df.merge(prod_map, left_on='제품명', right_on='prod_name', how='left')

reviews_insert = reviews_df[['prod_idx', '리뷰', '별점', 'ad_flag', '리뷰URL']].rename(columns={
    '리뷰': 'review_content',
    '별점': 'rating',
    'ad_flag': 'ad_flag',
    '리뷰URL': 'review_url'
})

# prod_idx 없는 행 제거
reviews_insert = reviews_insert.dropna(subset=['prod_idx'])
reviews_insert['prod_idx'] = reviews_insert['prod_idx'].astype(int)

reviews_insert.to_sql(
    't_review',
    con=engine,
    if_exists='append',
    index=False,
    chunksize=1000
)

# prod_idx 없는 행 제거
reviews_insert = reviews_insert.dropna(subset=['prod_idx'])
reviews_insert['prod_idx'] = reviews_insert['prod_idx'].astype(int)

# ↓ 이 두 줄 추가
reviews_insert = reviews_insert.dropna(subset=['prod_idx'])
reviews_insert['prod_idx'] = reviews_insert['prod_idx'].astype(int)

reviews_insert = reviews_insert.dropna(subset=['prod_idx'])
reviews_insert['prod_idx'] = reviews_insert['prod_idx'].astype(int)
reviews_insert['rating'] = reviews_insert['rating'].fillna(0)  # ← 이 줄 추가

print(f'✅ t_review 적재 완료: {len(reviews_insert)}개')

In [ ]:
# 셀 7: t_ingredient 적재 (제품별 성분을 행으로 분리)
# 전성분 예: "정제수, 글리세린, 나이아신아마이드" → 각각 1행씩

ingre_rows = []

# 제품별 전성분 (중복 제거)
prod_ingre = df[['제품명', '전성분']].drop_duplicates(subset='제품명')
prod_ingre = prod_ingre.merge(prod_map, left_on='제품명', right_on='prod_name', how='left')

for _, row in prod_ingre.iterrows():
    if pd.isna(row['전성분']) or row['전성분'] is None:
        continue
    ingredients = [i.strip() for i in str(row['전성분']).split(',') if i.strip()]
    for ingre in ingredients:
        ingre_rows.append({
            'prod_idx': int(row['prod_idx']),
            'ingre_name': ingre[:50],  # VARCHAR(50) 제한
            'allergy_check': None,     # 추후 업데이트
            'recommended_skin': None,  # GPT로 추후 업데이트
            'sensitive_skin': None     # GPT로 추후 업데이트
        })

ingre_df = pd.DataFrame(ingre_rows)

ingre_df.to_sql(
    't_ingredient',
    con=engine,
    if_exists='append',
    index=False,
    chunksize=1000
)
print(f'✅ t_ingredient 적재 완료: {len(ingre_df)}개 성분')

In [ ]:
# 셀 8: 적재 결과 확인
with engine.connect() as conn:
    for table in ['t_product', 't_review', 't_ingredient']:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()
        print(f'{table}: {count:,}행')

In [ ]:
# 먼저 NULL인 행 확인
print(products_insert[products_insert['brand_name'].isna()])

In [ ]:
# 현재 상태 확인
print(products_insert.isna().sum())
print(f'총 행 수: {len(products_insert)}')

In [ ]:
# 중복 확인
with engine.connect() as conn:
    dup = pd.read_sql('''
        SELECT prod_name, COUNT(*) as cnt 
        FROM t_product 
        GROUP BY prod_name 
        HAVING cnt > 1
        LIMIT 5
    ''', conn)
    print(f'중복 제품 수: {len(dup)}개')
    print(dup)

In [ ]:
# 전체 삭제 후 재적재
with engine.connect() as conn:
    conn.execute(text('DELETE FROM t_review'))
    conn.execute(text('DELETE FROM t_product'))
    conn.execute(text('ALTER TABLE t_product AUTO_INCREMENT = 1'))
    conn.execute(text('ALTER TABLE t_review AUTO_INCREMENT = 1'))
    conn.commit()

with engine.connect() as conn:
    for table in ['t_product', 't_review']:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()
        print(f'{table}: {count}행')
print('✅ 삭제 완료! 셀 4부터 다시 실행하세요.')

In [ ]:
# 현재 t_product 상태 확인
with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM t_product')).scalar()
    print(f't_product: {count}행')

In [ ]:
# t_review 비우고 다시 한 번만 적재
with engine.connect() as conn:
    conn.execute(text('DELETE FROM t_review'))
    conn.execute(text('ALTER TABLE t_review AUTO_INCREMENT = 1'))
    conn.commit()

with engine.connect() as conn:
    count = conn.execute(text('SELECT COUNT(*) FROM t_review')).scalar()
    print(f't_review: {count}행')
print('✅ 삭제 완료! 셀 6 한 번만 다시 실행하세요.')

In [ ]:
# 셀 8 실행
with engine.connect() as conn:
    for table in ['t_product', 't_review', 't_ingredient']:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()
        print(f'{table}: {count:,}행')

In [ ]:
# 셀 7 수정본
ingre_rows = []

prod_ingre = df[['제품명', '전성분']].drop_duplicates(subset='제품명')
prod_ingre = prod_ingre.merge(prod_map, left_on='제품명', right_on='prod_name', how='left')

# prod_idx 없는 행 제거
prod_ingre = prod_ingre.dropna(subset=['prod_idx'])
prod_ingre['prod_idx'] = prod_ingre['prod_idx'].astype(int)

for _, row in prod_ingre.iterrows():
    if pd.isna(row['전성분']) or row['전성분'] is None:
        continue
    ingredients = [i.strip() for i in str(row['전성분']).split(',') if i.strip()]
    for ingre in ingredients:
        ingre_rows.append({
            'prod_idx': row['prod_idx'],
            'ingre_name': ingre[:50],
            'allergy_check': None,
            'recommended_skin': None,
            'sensitive_skin': None
        })

ingre_df = pd.DataFrame(ingre_rows)
print(f'적재할 성분 수: {len(ingre_df)}개')

with engine.begin() as conn:
    ingre_df.to_sql(
        't_ingredient',
        con=conn,
        if_exists='append',
        index=False,
        chunksize=1000
    )
print(f'✅ t_ingredient 적재 완료: {len(ingre_df)}개')

In [ ]:
with engine.connect() as conn:
    ingre_list = pd.read_sql(
        'SELECT DISTINCT ingre_name FROM t_ingredient WHERE prod_idx IS NOT NULL', 
        conn
    )
print(f'고유 성분 수: {len(ingre_list)}개')